In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.sql import functions as F
from pyspark.sql import Row
catalog_name='ecommerce'

df_products=spark.table(f"{catalog_name}.silver.slv_products")
df_brands=spark.table(f"{catalog_name}.silver.slv_brands")
df_category=spark.table(f"{catalog_name}.silver.slv_category")


# create a view or table

df_products.createOrReplaceTempView("v_products")
df_brands.createOrReplaceTempView("v_brands") 
df_category.createOrReplaceTempView("v_category")


display(spark.sql("""select * from  v_products limit 5"""))

use catalog ecommerce

spark.sql(f"use catalog {catalog_name}")

In [0]:
%sql

CREATE or replace table gold.gld_dim_products AS

with brands_categories As(
    SELECT
    b.brand_name,
    b.brand_code,
    c.category_name,
    c.category_code
    from v_brands b 
    inner join v_category c 
    on b.category_code = c.category_code
)
SELECT
  d.product_id,
  d.sku,
  d.category_code,
  COALESCE(e.category_name, 'Not Available') AS category_name,
  d.brand_code,
  COALESCE(e.brand_name, 'Not Available')   AS brand_name,
  d.color,
  d.size,
  d.material,
  d.weight_grams,
  d.length_cm,
  d.width_cm,
  d.height_cm,
  d.rating_count,
  d.file_name,
  d.ingest_time
FROM v_products d
LEFT JOIN brands_categories e
  ON d.brand_code = e.brand_code;

In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  

country_state_map

In [0]:
# 1 Flatten country_state_map into a list of Rows
rows = []
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))
rows[:10]        

In [0]:
# 2️ Create mapping DataFrame
df_region_mapping = spark.createDataFrame(rows)

# Optional: show mapping
df_region_mapping.show(truncate=False)


df_silver = spark.table(f'{catalog_name}.silver.slv_customers')
df_gold = df_silver.join(df_region_mapping, on=['country', 'state'], how='left')

df_gold = df_gold.fillna({'region': 'Other'})

df_gold.write.format('delta')\
    .mode('overwrite')\
    .option('overwriteSchema', 'true')\
    .saveAsTable(f'{catalog_name}.gold.gld_dim_customers')

In [0]:
df_date=spark.table(f"{catalog_name}.silver.slv_date")

df_date = df_date.withColumn("date_id", F.date_format(F.col("date"),"yyyyMMdd").cast("int"))

# Add month name (e.g., 'January', 'February', etc.)
df_gold = df_date.withColumn("month_name", F.date_format(F.col("date"), "MMMM"))

df_gold=df_gold.withColumn(
    "is_weekend",F.when(F.col('day_name').isin('Saturday','Sunday'),1).otherwise(0)
    )
desired_columns_order = ["date_id", "date", "year", "month_name", "day_name", "is_weekend", "quater", "week", "ingest_time", "file_name"]

df_gold = df_gold.select(desired_columns_order)

# write table to gold layer
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_dim_date")